In [1]:
import sys
sys.path.append("..")
import ee
ee.Initialize(project="tree-fire-510209")

from src.fires import load_fire
from src.grids import pad_bounds

uid = "2020_135"   # already documented steep-terrain case, README known issue #8
fire = load_fire(uid)
bounds = pad_bounds(fire.total_bounds, 0.15)
aoi = ee.Geometry.Rectangle(bounds)

dem = ee.Image("USGS/SRTMGL1_003")
slope = ee.Terrain.slope(dem).clip(aoi)

# just check feasibility and get a quick summary first, no export yet
stats = slope.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=aoi, scale=100, maxPixels=1e9,
).getInfo()
print(stats)

{'slope_max': 62.34728068438627, 'slope_mean': 14.717951139737979, 'slope_min': 0}


In [ ]:
# import subprocess
# print(subprocess.run(["ls", "../data/cci_agb_ee/"], capture_output=True, text=True).stdout)

In [3]:
import subprocess

GCS_BUCKET = "tree-fire-510209-data"
GCS_PREFIX = "fire-impact-curves/data/terrain"

task = ee.batch.Export.image.toCloudStorage(
    image=slope,
    description=f"slope_{uid}",
    bucket=GCS_BUCKET,
    fileNamePrefix=f"{GCS_PREFIX}/slope_{uid}",
    region=aoi,
    scale=100,
    maxPixels=1e9,
)
task.start()
print("submitted, task id:", task.id)

import time
while task.active():
    print(task.status()["state"])
    time.sleep(30)
print("final state:", task.status()["state"])

if task.status()["state"] == "COMPLETED":
    import os
    os.makedirs("../data/terrain", exist_ok=True)
    out_path = f"../data/terrain/slope_{uid}.tif"
    subprocess.run(["gsutil", "cp", f"gs://{GCS_BUCKET}/{GCS_PREFIX}/slope_{uid}.tif", out_path], check=True)
    print("downloaded ->", out_path)

submitted, task id: FPUIA4GAC2D7SNXMX4VZ7QUR
READY
final state: COMPLETED


Copying gs://tree-fire-510209-data/fire-impact-curves/data/terrain/slope_2020_135.tif...
/ [1 files][  3.1 MiB/  3.1 MiB]                                                
Operation completed over 1 objects/3.1 MiB.                                      


downloaded -> ../data/terrain/slope_2020_135.tif


In [4]:
import numpy as np
import pandas as pd
from rasterio.warp import Resampling
from rasterio.features import rasterize
from src.grids import read_window, reproject_to_grid, pad_bounds
from src.fires import inner_burn_area, utm_epsg_for
from src.height import height_mosaic_for

lon, lat = fire.geometry.iloc[0].centroid.x, fire.geometry.iloc[0].centroid.y
utm_epsg = utm_epsg_for(lon, lat)
inner = inner_burn_area(fire, utm_epsg=utm_epsg)

agb_before, tr_ref, crs_ref = read_window("../data/cci_agb_ee/agb_2020_135_2019.tif", bounds)
shape = agb_before.shape

mask_inner = rasterize(
    [(g, 1) for g in inner.geometry], out_shape=shape,
    transform=tr_ref, fill=0, dtype="uint8",
).astype(bool)
valid = np.isfinite(agb_before) & (agb_before > 10)
keep = mask_inner & valid

height_path = height_mosaic_for(lon, lat)
height_arr, tr_h, crs_h = read_window(height_path, bounds, masked=False)
height_100m = reproject_to_grid(
    height_arr.astype("float32"), tr_h, crs_h, shape, tr_ref, crs_ref,
    resampling=Resampling.average,
)

slope_arr, tr_s, crs_s = read_window("../data/terrain/slope_2020_135.tif", bounds)
slope_100m = reproject_to_grid(
    np.nan_to_num(slope_arr, nan=-9999).astype("float32"), tr_s, crs_s, shape, tr_ref, crs_ref,
    resampling=Resampling.average, src_nodata=-9999,
)

d = pd.DataFrame({
    "height": height_100m[keep],
    "slope": slope_100m[keep],
    "agb_before": agb_before[keep],
})

# the specific documented anomaly: short height (<10m) but implausibly high biomass
d["implausible"] = (d["height"] < 10) & (d["agb_before"] > 200)

print(d.groupby("implausible")[["slope", "height", "agb_before"]].describe().T)

implausible               False        True 
slope      count  109641.000000  1460.000000
           mean       16.383614    27.369686
           std         8.932633     7.241906
           min         0.583210     3.168596
           25%         9.396637    23.068799
           50%        14.583961    28.314126
           75%        22.143894    32.525886
           max        57.979603    44.661201
height     count  109641.000000  1460.000000
           mean       20.363340     6.036913
           std         7.731354     2.820734
           min         0.000000     0.000000
           25%        15.190033     3.907000
           50%        21.638403     6.575956
           75%        26.375055     8.452321
           max       101.000000     9.996759
agb_before count  109641.000000  1460.000000
           mean      161.031868   260.342466
           std       102.314683    49.375808
           min        11.000000   201.000000
           25%        76.000000   221.000000
          

In [6]:
import subprocess, time, os

def export_and_download_slope(uid):
    fire = load_fire(uid)
    bounds = pad_bounds(fire.total_bounds, 0.15)
    aoi = ee.Geometry.Rectangle(bounds)
    dem = ee.Image("USGS/SRTMGL1_003")
    slope = ee.Terrain.slope(dem).clip(aoi)

    task = ee.batch.Export.image.toCloudStorage(
        image=slope, description=f"slope_{uid}",
        bucket="tree-fire-510209-data",
        fileNamePrefix=f"fire-impact-curves/data/terrain/slope_{uid}",
        region=aoi, scale=100, maxPixels=1e9,
    )
    task.start()
    print(uid, "submitted")

    while task.active():
        time.sleep(30)
    print(uid, "final state:", task.status()["state"])

    if task.status()["state"] == "COMPLETED":
        os.makedirs("../data/terrain", exist_ok=True)
        out_path = f"../data/terrain/slope_{uid}.tif"
        subprocess.run(["gsutil", "cp",
                         f"gs://tree-fire-510209-data/fire-impact-curves/data/terrain/slope_{uid}.tif",
                         out_path], check=True)
        print(uid, "downloaded ->", out_path)
        return out_path
    return None

from src.fires import load_fire
from src.grids import pad_bounds

In [7]:
slope_path_112 = export_and_download_slope("2020_112")
slope_path_841874 = export_and_download_slope("2020_841874")

2020_112 submitted
2020_112 final state: COMPLETED


Copying gs://tree-fire-510209-data/fire-impact-curves/data/terrain/slope_2020_112.tif...
/ [1 files][  2.8 MiB/  2.8 MiB]                                                
Operation completed over 1 objects/2.8 MiB.                                      


2020_112 downloaded -> ../data/terrain/slope_2020_112.tif
2020_841874 submitted
2020_841874 final state: COMPLETED


Copying gs://tree-fire-510209-data/fire-impact-curves/data/terrain/slope_2020_841874.tif...
/ [1 files][  1.1 MiB/  1.1 MiB]                                                
Operation completed over 1 objects/1.1 MiB.                                      


2020_841874 downloaded -> ../data/terrain/slope_2020_841874.tif


In [8]:
import numpy as np
import pandas as pd
from rasterio.warp import Resampling
from rasterio.features import rasterize
from src.grids import read_window, reproject_to_grid
from src.fires import inner_burn_area, utm_epsg_for
from src.height import height_mosaic_for

def slope_vs_implausible_height(uid, agb_before_year, slope_path, min_agb_before=10, implausible_agb=200):
    fire = load_fire(uid)
    bounds = pad_bounds(fire.total_bounds, 0.15)
    lon, lat = fire.geometry.iloc[0].centroid.x, fire.geometry.iloc[0].centroid.y
    utm_epsg = utm_epsg_for(lon, lat)
    inner = inner_burn_area(fire, utm_epsg=utm_epsg)

    agb_path = f"../data/cci_agb_ee/agb_{uid}_{agb_before_year}.tif"
    agb_before, tr_ref, crs_ref = read_window(agb_path, bounds)
    shape = agb_before.shape

    mask_inner = rasterize(
        [(g, 1) for g in inner.geometry], out_shape=shape,
        transform=tr_ref, fill=0, dtype="uint8",
    ).astype(bool)
    valid = np.isfinite(agb_before) & (agb_before > min_agb_before)
    keep = mask_inner & valid

    height_path = height_mosaic_for(lon, lat)
    height_arr, tr_h, crs_h = read_window(height_path, bounds, masked=False)
    height_100m = reproject_to_grid(
        height_arr.astype("float32"), tr_h, crs_h, shape, tr_ref, crs_ref,
        resampling=Resampling.average,
    )

    slope_arr, tr_s, crs_s = read_window(slope_path, bounds)
    slope_100m = reproject_to_grid(
        np.nan_to_num(slope_arr, nan=-9999).astype("float32"), tr_s, crs_s, shape, tr_ref, crs_ref,
        resampling=Resampling.average, src_nodata=-9999,
    )

    d = pd.DataFrame({
        "height": height_100m[keep], "slope": slope_100m[keep], "agb_before": agb_before[keep],
    })
    d["implausible"] = (d["height"] < 10) & (d["agb_before"] > implausible_agb)

    summary = d.groupby("implausible")["slope"].agg(["count", "mean", "median"])
    print(f"=== {uid} ===")
    print(summary)
    print()
    return d

In [9]:
d1 = slope_vs_implausible_height("2020_112", agb_before_year=2019, slope_path=slope_path_112)
d2 = slope_vs_implausible_height("2020_841874", agb_before_year=2019, slope_path=slope_path_841874)

=== 2020_112 ===
              count       mean     median
implausible                              
False        125097  18.156551  18.093311
True             36  22.830666  24.688648

=== 2020_841874 ===
             count       mean     median
implausible                             
False         1541  18.170429  17.686668
True            50  14.928025  15.325512

